In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

import mlrose_hiive
from mlrose_hiive import QueensGenerator
from mlrose_hiive import SARunner

In [2]:
EXPERIMENT_NAME = 'SA_Queens_GridSearch'
SEED = 0
PROBLEM_SIZE_LIST = [1000]
ITERATIONS_LIST = [1e9]
# MAX_ATTEMPTS_LIST = [1, 10, 100, 1000, 10000]
MAX_ATTEMPTS_LIST = [1000]
# TEMPERATURE_LIST = [0.001, 0.1, 1, 10, 100, 1000]
TEMPERATURE_LIST = [1e3, 1e4, 1e5]
DECAY_LIST = {
    'GeomDecay': mlrose_hiive.GeomDecay, 
    # 'ExpDecay': mlrose_hiive.ExpDecay, 
    # 'ArithDecay': mlrose_hiive.ArithDecay,
}
NUM_RUNS = 3

In [3]:
output_dir = f'metrics/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    for max_iterations in ITERATIONS_LIST:
        for max_attempts in MAX_ATTEMPTS_LIST:
            for start_temperature in tqdm(TEMPERATURE_LIST):
                for decay_str, decay_cls in DECAY_LIST.items():
                    for i in range(NUM_RUNS):
                        problem = QueensGenerator().generate(seed=SEED+i, size=problem_size)
                        sa = SARunner(
                            problem=problem,
                            experiment_name='dontcare',
                            output_directory='/Users/sdale/temp',
                            seed=SEED+i,
                            iteration_list=[max_iterations],
                            max_attempts=max_attempts,
                            temperature_list=[start_temperature],
                            decay_list=[decay_cls],
                        )
                        _, df_run_curves = sa.run()
                        df_run_curves['group_number'] = group_i
                        df_run_curves['run_number'] = run_i
                        df_run_curves['problem_size'] = problem_size
                        df_run_curves['max_iterations'] = max_iterations
                        df_run_curves['max_attempts'] = max_attempts
                        df_run_curves['start_temperature'] = start_temperature
                        df_run_curves['decay_type'] = decay_str

                        all_df = pd.concat([all_df, df_run_curves], axis=0)
                        run_i += 1
                    group_i += 1
all_df.reset_index(inplace=True, drop=True)

 67%|██████▋   | 2/3 [26:16:29<14:17:59, 51479.16s/it]

In [ ]:
all_df['Fitness'] = -1 * all_df['Fitness']

In [ ]:
all_df['Fitness'].max()

In [ ]:
all_df.to_csv(os.path.join(output_dir, 'learning_curve.csv'), index=False)